<a href="https://colab.research.google.com/github/Vishali-Vivek006/Engineering-employability-prediction/blob/main/notebooks/03_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Upload the CSV file
from google.colab import files
uploaded = files.upload()

# Read the uploaded CSV
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset shape:", df.shape)
print("Target distribution:")
print(df["Placement_Status"].value_counts())

Saving Employability_dataset.csv to Employability_dataset.csv
Dataset shape: (5000, 157)
Target distribution:
Placement_Status
0    2872
1    2128
Name: count, dtype: int64


In [2]:
# Target must contain only 0 and 1
assert df["Placement_Status"].notna().all(), \
    "Placement_Status contains missing values."

assert set(df["Placement_Status"].unique()).issubset({0, 1}), \
    "Unexpected target labels found."

print("0 = Not placed")
print("1 = Placed")
print("\nPlacement percentages:")
print((df["Placement_Status"].value_counts(normalize=True) * 100).round(2))

0 = Not placed
1 = Placed

Placement percentages:
Placement_Status
0    57.44
1    42.56
Name: proportion, dtype: float64


In [3]:
target = "Placement_Status"

columns_to_drop = [
    "Student_ID",
    "Data_Source",
    "UG_Required"
]

# Drop only columns that actually exist
columns_to_drop = [c for c in columns_to_drop if c in df.columns]

X = df.drop(columns=[target] + columns_to_drop)
y = df[target].astype(int)

print("Dropped columns:", columns_to_drop)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nRemaining missing values:")
print(X.isnull().sum().sort_values(ascending=False).head(10))

Dropped columns: ['Student_ID', 'Data_Source', 'UG_Required']
Feature matrix shape: (5000, 153)
Target shape: (5000,)

Remaining missing values:
PG_Specialization              4125
Education_Level                   0
UG_Degree                         0
PG_Required_For_Target_Role       0
Relevant_Degree_Required          0
Branch                            0
Year                              0
CGPA                              0
Attendance_Percentage             0
Backlogs                          0
dtype: int64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).round(3))

Training features: (4000, 153)
Testing features: (1000, 153)

Training target distribution:
Placement_Status
0    0.574
1    0.426
Name: proportion, dtype: float64

Testing target distribution:
Placement_Status
0    0.574
1    0.426
Name: proportion, dtype: float64


In [5]:
categorical_cols = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Number of categorical features:", len(categorical_cols))
print("Number of numerical features:", len(numerical_cols))

print("\nCategorical columns:")
print(categorical_cols)

print("\nNumerical columns:", len(numerical_cols))

Number of categorical features: 4
Number of numerical features: 149

Categorical columns:
['Education_Level', 'UG_Degree', 'PG_Specialization', 'Branch']

Numerical columns: 149


In [6]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"
    )),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ],
    remainder="drop"
)

# Fit on training data only
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the learned transformations to test data
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing completed.")
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Preprocessing completed.
Processed training shape: (4000, 169)
Processed testing shape: (1000, 169)


In [7]:
print("Missing values in original training data:", X_train.isnull().sum().sum())

print("Processed training data contains missing values:",
      np.isnan(X_train_processed.data).any()
      if hasattr(X_train_processed, "data")
      else np.isnan(X_train_processed).any())

print("Processed testing data contains missing values:",
      np.isnan(X_test_processed.data).any()
      if hasattr(X_test_processed, "data")
      else np.isnan(X_test_processed).any())

print("\nPreprocessing verification completed.")

Missing values in original training data: 3299
Processed training data contains missing values: False
Processed testing data contains missing values: False

Preprocessing verification completed.


In [8]:
import joblib
from scipy import sparse

# Save processed matrices
if sparse.issparse(X_train_processed):
    sparse.save_npz("X_train_processed.npz", X_train_processed)
    sparse.save_npz("X_test_processed.npz", X_test_processed)
else:
    np.save("X_train_processed.npy", X_train_processed)
    np.save("X_test_processed.npy", X_test_processed)

y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

# Save the fitted preprocessor for reuse during prediction
joblib.dump(preprocessor, "preprocessor.joblib")

print("Saved preprocessing outputs successfully.")

Saved preprocessing outputs successfully.
